# Train on a free Colab T4, then bring the results back to the GitHub repo

Colab only provides the GPU. The deliverable stays the GitHub repository: this notebook clones it, trains, and packs everything
you need to commit into **one zip** that you download and unzip into your local copy of the repo.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`. Each training run takes roughly 30 minutes (my estimate; watch `tokens_per_sec`); there are four (small/full data x English/Amharic), so this may need two Colab sessions: download the zip after each.
Colab's disk is wiped when the session ends, so download the zip (last cell) before closing the tab.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv   # should say Tesla T4

## 1. Get the code
If the repository is private, use `https://<your-token>@github.com/Efrata-Habte/GPT-2.git` (a GitHub personal access token with read access).

In [ ]:
!git clone https://github.com/Efrata-Habte/GPT-2.git
%cd GPT-2
!pip install -q -r requirements.txt

## 2. Data

In [ ]:
!python data/prepare.py                       # Tiny Shakespeare (small English data)
!python -I data/amharic/prepare.py            # MasakhaNEWS Amharic (small Amharic data)
!python -I data/prepare_full_shakespeare.py   # Complete Works of Shakespeare (full English data) + shared English test set
!python -I data/amharic/prepare_wikipedia.py  # Amharic Wikipedia + news (full Amharic data) + shared Amharic test sets

## 3. Train the assignment model (English) and the Amharic model
If Colab disconnects, re-run the clone/install cells and use the *resume* cell below.

In [ ]:
!python -m src.train --config configs/small.yaml

In [ ]:
!python -m src.train --config configs/amharic.yaml

**Full-data models** (same architecture and steps, ~3-4x more text): the demo's *Small vs full data* tab compares them with the two above.

In [ ]:
!python -m src.train --config configs/small_full.yaml
!python -m src.train --config configs/amharic_full.yaml

**Resume after a disconnect** (only if a run was interrupted; needs the checkpoint folder still on disk or restored from Drive):

In [ ]:
# !python -m src.train --config configs/small.yaml --resume checkpoints/small/ckpt_last.pt

## 4. Experiments (optional, but this is where the GPU helps most)
Experiments 1, 2 and 9 take a few minutes; each sweep (3-8) roughly 30-40 minutes. Run as many as your session allows.

In [ ]:
!python experiments/exp1_temperature.py --ckpt checkpoints/small/ckpt_best.pt
!python experiments/exp2_context.py --ckpt checkpoints/small/ckpt_best.pt
!python experiments/exp9_attention.py --ckpt checkpoints/small/ckpt_best.pt
!python experiments/exp3_dynamics.py --log checkpoints/small/log.csv     # analysis of the run above (free); without --log it trains configs/exp_overfit.yaml
!python experiments/exp12_data_scale.py --english_small checkpoints/small/ckpt_best.pt --english_full checkpoints/small_full/ckpt_best.pt --amharic_small checkpoints/amharic/ckpt_best.pt --amharic_full checkpoints/amharic_full/ckpt_best.pt   # small vs full data

In [ ]:
# Sweeps: uncomment what you have time for
# !python experiments/exp4_architecture.py --param n_layer
# !python experiments/exp5_norm.py
# !python experiments/exp6_pos_encoding.py
# !python experiments/exp7_tokenizer.py
# !python experiments/exp8_scaling.py --extra_runs results/runs/exp4_n_layer
# !python experiments/exp11_amharic_study.py --runs "English (small)=checkpoints/small" "Amharic (small)=checkpoints/amharic" --sample_run checkpoints/amharic

## 5. Pack the results
The slim exported models (weights + tokenizer, no optimizer state) are small enough for GitHub; the full training checkpoints are not needed in the repo.

In [ ]:
!python scripts/export_for_demo.py --english checkpoints/small/ckpt_best.pt --amharic checkpoints/amharic/ckpt_best.pt \n    --english_full checkpoints/small_full/ckpt_best.pt --amharic_full checkpoints/amharic_full/ckpt_best.pt
!python experiments/make_summary.py
!zip -qr trained_outputs.zip app/checkpoints results checkpoints/*/log.csv checkpoints/*/summary.json checkpoints/*/loss_curve.png checkpoints/*/meta.json
!du -h trained_outputs.zip

In [ ]:
from google.colab import files
files.download("trained_outputs.zip")

## 6. Back on your laptop
1. Unzip `trained_outputs.zip` **into the repository folder** (allow it to overwrite).
2. `python -m src.generate --ckpt app/checkpoints/english/model.pt --prompt "ROMEO:"` to check it works, and `streamlit run app/app.py` for the demo.
3. Commit and push:
   ```
   git add app/checkpoints results checkpoints/small checkpoints/amharic checkpoints/small_full checkpoints/amharic_full
   git commit -m "Add T4-trained small and Amharic models and results"
   git push
   ```